In [ ]:
import hashlib
import importlib
import json
import platform
import shutil
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

started_notebook = time.perf_counter()

def is_repository(path):
    return all(
        (path / name).is_file()
        for name in (
            "pyproject.toml",
            "configs/research_config.example.json",
            "src/eurusd_research/history.py",
        )
    )

candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p.resolve() for p in candidates if is_repository(p)), None)

if REPO_ROOT is None:
    entered = input("Paste the full local repository folder path: ").strip().strip('"')
    REPO_ROOT = Path(entered).expanduser().resolve()

if not is_repository(REPO_ROOT):
    raise RuntimeError(f"Repository is incomplete or incorrect: {REPO_ROOT}")

packages = [
    "numpy", "pandas", "pyarrow", "duckdb", "scikit-learn",
    "scipy", "pytest", "matplotlib", "tzdata", "pytz",
]
VERSIONS = {}
for name in packages:
    try:
        VERSIONS[name] = version(name)
    except PackageNotFoundError:
        VERSIONS[name] = None

print("Repository:", REPO_ROOT)
print("Python:", platform.python_version())
print("Kernel executable:", sys.executable)
print("Operating system:", platform.system(), platform.release())
print("Packages:", json.dumps(VERSIONS, indent=2))

if any(value is None for value in VERSIONS.values()):
    raise RuntimeError("Required dependencies are missing. Stop and return this output.")

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from eurusd_research.config import load_config
from eurusd_research import features as feature_io
from eurusd_research import history
from eurusd_research.data import _sha256

config = load_config(REPO_ROOT)

planned_files = [
    "src/eurusd_research/completion_data.py",
    "src/eurusd_research/completion_models.py",
    "src/eurusd_research/completion_evaluation.py",
    "tests/test_completion.py",
    "README.md",
    "CONTRIBUTIONS.md",
    "reports/research_results.md",
]

print("\nFiles inspected before writing:")
for relative in planned_files:
    path = REPO_ROOT / relative
    print(relative, "EXISTS" if path.exists() else "MISSING")

def put_source(relative, content):
    """Create new source, or accept an identical existing implementation."""
    path = REPO_ROOT / relative
    content = content.lstrip()
    if path.exists():
        if path.read_text(encoding="utf-8") != content:
            raise RuntimeError(
                f"Existing file differs: {relative}. "
                "Stop rather than overwrite existing work."
            )
        print("UNCHANGED", relative)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8", newline="\n") as handle:
            handle.write(content)
        print("CREATED", relative)

In [ ]:
history_manifest = REPO_ROOT / config["active_history_manifest"]
history_snapshot = json.loads(history_manifest.read_text(encoding="utf-8"))
reservation = config["full_history_validation_plan"]

if history_snapshot["job_id"] != reservation["source_collection_id"]:
    raise RuntimeError("History collection and final-test reservation differ.")

FEATURE_SETTINGS = dict(config["sampling_features"])
MODEL_SETTINGS = dict(config["baseline_models"])
REGIME_SETTINGS = dict(config["liquidity_regimes"])

if (
    FEATURE_SETTINGS["max_staleness_seconds"] != 30
    or FEATURE_SETTINGS["substantial_gap_seconds"] != 30
    or MODEL_SETTINGS["ridge_alpha"] != 10.0
):
    raise RuntimeError("Configuration differs from the agreed research policy.")

pilot_period = next(
    item for item in config["pilot_validation_plan"]["periods"]
    if item["role"] == "pilot_holdout"
)

PROTOCOL = {
    "version": "completion_v1",
    "source_collection_id": history_snapshot["job_id"],
    "source_manifest_sha256": _sha256(history_manifest),
    "feature_settings": FEATURE_SETTINGS,
    "model_settings": {
        "price_features": MODEL_SETTINGS["price_features"],
        "size_features": MODEL_SETTINGS["size_features"],
        "required_current_size_features": MODEL_SETTINGS["required_current_size_features"],
        "ridge_alpha": 10.0,
    },
    "regime_settings": REGIME_SETTINGS,
    "horizons_seconds": [1, 5, 30, 60],
    "training_calendar_days": 365,
    "imputation": "training-only mean; indicators for every predictor",
    "regime_threshold_sample_seconds": 300,
    "forecast_bucket_sample_seconds": 300,
    "training_sample_purpose": "thresholds and bucket boundaries only; Ridge uses all eligible rows",
    "development_start_utc": "2017-01-01T00:00:00+00:00",
    "development_end_utc": reservation["development"]["end_utc_exclusive"],
    "final_test": reservation["reserved_final_test"],
    "excluded_signal_and_label_intervals": [{
        "start": pilot_period["start_utc"],
        "end": pilot_period["end_utc"],
        "reason": "earlier protected January pilot holdout",
    }],
    "bootstrap_block_dates": [5, 20],
    "bootstrap_repetitions": 1000,
    "confidence": 0.95,
    "random_seed": 42,
    "minimum_fit_rows": 1000,
    "maximum_training_sample_rows": 120000,
    "maximum_observed_frame_mib": 512,
    "minimum_free_gib_during_run": 16,
    "final_test_policy": "fixed choices; no parameter or feature selection from final results",
    "h001_status": "untested; optional future development experiment",
    "versions": VERSIONS,
}

ignore_probes = [
    "data/processed/completion/probe.parquet",
    "artifacts/manifests/completion/probe.json",
    "artifacts/forecasts/completion/probe.parquet",
    "reports/private/completion/probe.json",
    "configs/research_config.local.json",
]

for relative in ignore_probes:
    checked = subprocess.run(
        ["git", "check-ignore", "--quiet", relative],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )
    if checked.returncode != 0:
        raise RuntimeError(f"Generated/private path is not ignored: {relative}")

print("PASS: generated data, artifacts, private reports, and local configuration are ignored.")
print(json.dumps(PROTOCOL, indent=2))

In [ ]:
import importlib
import os
import tempfile
from pathlib import Path

from eurusd_research import features as feature_io

module_path = REPO_ROOT / "src/eurusd_research/completion_data.py"
test_path = REPO_ROOT / "tests/test_completion_initial_coverage.py"

if not module_path.is_file():
    raise RuntimeError("completion_data.py is missing. Return this output.")

original = module_path.read_text(encoding="utf-8")

old_block = '''    if not selected:
        raise RuntimeError("No source partitions available for the requested block.")
'''

new_block = '''    if not selected:
        if stop == 0 and index["parts"]:
            # Before the first source observation, preserve an unavailable
            # clock interval using schema only. Do not read future quote values.
            first_path = safe_path(root, index["parts"][0]["path"])
            source_schema = pq.ParquetFile(first_path).schema_arrow
            empty_schema = pa.schema([
                source_schema.field(name) for name in io.QUOTE_COLUMNS
            ])
            return pa.Table.from_batches(
                [], schema=empty_schema
            ).to_pandas()
        raise RuntimeError("No source partitions available for the requested block.")
'''

updated = original

if new_block not in updated:
    if updated.count(old_block) != 1:
        raise RuntimeError(
            "The loader differs from the expected implementation. "
            "Stop and return this output."
        )
    updated = updated.replace(old_block, new_block, 1)

if "import pyarrow as pa\n" not in updated:
    marker = "import pyarrow.parquet as pq\n"
    if updated.count(marker) != 1:
        raise RuntimeError("Could not locate the expected PyArrow import.")
    updated = updated.replace(
        marker, "import pyarrow as pa\n" + marker, 1
    )

regression_tests = r'''
"""Regression checks for intervals without source observations."""

import pandas as pd

from test_completion import policy, quotes
from eurusd_research import features as io
from eurusd_research.completion_data import (
    assemble_block, read_quotes, source_index,
)


def source_fixture(tmp_path):
    q = quotes()
    path = tmp_path / "quotes.parquet"
    q[io.QUOTE_COLUMNS].to_parquet(path, index=False)
    collection = {
        "parts": [{
            "path": "quotes.parquet",
            "min_utc": q["timestamp_utc"].iloc[0].isoformat(),
            "max_utc": q["timestamp_utc"].iloc[-1].isoformat(),
        }]
    }
    return q, collection, source_index(collection)


def test_before_first_quote_remains_unavailable(tmp_path):
    q, collection, index = source_fixture(tmp_path)
    protocol = policy()
    start = q["timestamp_utc"].iloc[0] - pd.Timedelta(hours=1)
    end = start + pd.Timedelta(seconds=3)

    empty = read_quotes(
        tmp_path, collection, index, start, end,
        protocol["feature_settings"], tmp_path / "working",
    )

    assert empty.empty
    assert list(empty.columns) == io.QUOTE_COLUMNS
    assert str(empty["timestamp_utc"].dt.tz) == "UTC"

    output, _ = assemble_block(
        empty, start, end, protocol, q["timestamp_utc"].iloc[0]
    )
    assert len(output) == 3
    assert output["source_timestamp_utc"].isna().all()
    assert output["state_reason"].eq("no_source_quote").all()
    assert not output["price_valid"].any()
    for h in protocol["horizons_seconds"]:
        assert not output[f"label_valid_{h}s"].any()


def test_gap_after_history_retains_only_past_predecessor(tmp_path):
    q, collection, index = source_fixture(tmp_path)
    protocol = policy()
    settings = {
        **protocol["feature_settings"],
        "max_quotes_per_block": 10000,
        "duckdb_memory_limit": "64MB",
        "duckdb_temp_limit": "1GB",
    }
    start = q["timestamp_utc"].iloc[-1] + pd.Timedelta(hours=1)
    end = start + pd.Timedelta(seconds=3)

    previous = read_quotes(
        tmp_path, collection, index, start, end,
        settings, tmp_path / "working",
    )

    assert len(previous) == 1
    assert previous["timestamp_utc"].iloc[0] == q["timestamp_utc"].iloc[-1]
    assert previous["timestamp_utc"].lt(start).all()

    output, _ = assemble_block(
        previous, start, end, protocol, q["timestamp_utc"].iloc[0]
    )
    assert not output["price_valid"].any()
    assert output["state_reason"].eq("stale_snapshot").all()
'''.lstrip()

if test_path.exists():
    if test_path.read_text(encoding="utf-8") != regression_tests:
        raise RuntimeError(
            "Existing regression test differs. Stop rather than overwrite it."
        )

# Check syntax before replacing either file.
compile(updated, str(module_path), "exec")
compile(regression_tests, str(test_path), "exec")

def replace_text_atomically(path, content):
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            newline="\n",
            dir=path.parent,
            prefix=f".{path.name}.",
            suffix=".tmp",
            delete=False,
        ) as handle:
            temporary = Path(handle.name)
            handle.write(content)
            handle.flush()
            os.fsync(handle.fileno())
        feature_io._replace_path(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

if updated != original:
    replace_text_atomically(module_path, updated)
    print("UPDATED: completion_data.py")
else:
    print("UNCHANGED: loader correction already present")

if not test_path.exists():
    replace_text_atomically(test_path, regression_tests)
    print("CREATED: tests/test_completion_initial_coverage.py")
else:
    print("UNCHANGED: regression tests already present")

importlib.invalidate_caches()
from eurusd_research import completion_data
completion_data = importlib.reload(completion_data)

print("Correction installed. Validation is pending.")
print("Rerun Cell 12, then Cell 14, then Cell 16.")

put_source("src/eurusd_research/completion_models.py", r'''
"""Training-only preprocessing and bounded sufficient-statistic Ridge fits."""

import numpy as np
import pandas as pd

from .completion_data import eligible, frames
from .regimes import METRIC_COLUMNS


def new_statistics(width):
    return {
        "n": 0, "mean": np.zeros(width), "y_mean": 0.0,
        "gram": np.zeros((width, width)), "cross": np.zeros(width),
    }


def update_statistics(state, x, y):
    if len(y) == 0:
        return
    x, y = np.asarray(x, float), np.asarray(y, float)
    n = len(y)
    mean, y_mean = x.mean(axis=0), float(y.mean())
    centered, centered_y = x - mean, y - y_mean
    gram, cross = centered.T @ centered, centered.T @ centered_y
    old = state["n"]
    if old:
        weight = old * n / (old + n)
        delta = mean - state["mean"]
        gram += np.outer(delta, delta) * weight
        cross += delta * (y_mean - state["y_mean"]) * weight
    state["gram"] += gram
    state["cross"] += cross
    state["mean"] += (mean - state["mean"]) * n / (old + n)
    state["y_mean"] += (y_mean - state["y_mean"]) * n / (old + n)
    state["n"] += n


def transform(values, fill):
    values = np.asarray(values, float)
    missing = ~np.isfinite(values)
    return np.column_stack([np.where(missing, fill, values), missing.astype(float)])


def finish_statistics(state, fill, indices, columns, alpha):
    indices = np.asarray(indices, dtype=int)
    variance = np.maximum(np.diag(state["gram"])[indices] / state["n"], 0)
    scale = np.sqrt(variance)
    scale[scale <= 1e-12] = 1.0
    gram = state["gram"][np.ix_(indices, indices)] / np.outer(scale, scale)
    gram.flat[::len(indices) + 1] += alpha
    beta = np.linalg.solve(gram, state["cross"][indices] / scale)
    if not np.isfinite(beta).all():
        raise RuntimeError("Nonfinite Ridge coefficients.")
    return {
        "columns": columns, "fill": np.asarray(fill).tolist(),
        "indices": indices.tolist(),
        "mean": state["mean"][indices].tolist(),
        "scale": scale.tolist(), "beta": beta.tolist(),
        "intercept": float(state["y_mean"]), "training_rows": state["n"],
        "alpha": alpha,
    }


def predict(model, frame):
    values = frame[model["columns"]].to_numpy(dtype=float)
    x = transform(values, np.asarray(model["fill"]))[:, model["indices"]]
    result = (
        (x - np.asarray(model["mean"])) / np.asarray(model["scale"])
    ) @ np.asarray(model["beta"]) + model["intercept"]
    if not np.isfinite(result).all():
        raise RuntimeError("Nonfinite Ridge forecast.")
    return result


def fit_fold(root, dataset, start, end, protocol):
    settings = protocol["model_settings"]
    columns = settings["price_features"] + settings["size_features"]
    width = len(columns)
    horizons = protocol["horizons_seconds"]
    sums = {h: np.zeros(width) for h in horizons}
    counts = {h: np.zeros(width, dtype=np.int64) for h in horizons}
    rows = {h: 0 for h in horizons}
    maximum_targets = {h: None for h in horizons}
    sample_parts = []
    maximum_frame_bytes = 0

    # First pass: exact finite-value means on each horizon's matched population.
    for _, frame in frames(root, dataset, start, end):
        maximum_frame_bytes = max(
            maximum_frame_bytes,
            int(frame.memory_usage(index=True, deep=True).sum()),
        )
        values = frame[columns].to_numpy(dtype=float)
        for h in horizons:
            _, common = eligible(frame, h, start, end, protocol)
            selected = values[common.to_numpy()]
            finite = np.isfinite(selected)
            sums[h] += np.where(finite, selected, 0).sum(axis=0)
            counts[h] += finite.sum(axis=0)
            rows[h] += len(selected)
            if common.any():
                target = frame.loc[common, f"target_timestamp_utc_{h}s"].max()
                maximum_targets[h] = max(
                    maximum_targets[h] or target, target
                )

        times = frame["feature_timestamp_utc"]
        sample_mask = times.dt.second.eq(0) & times.dt.minute.mod(5).eq(0)
        sample_parts.append(frame.loc[sample_mask].copy())

    if any(rows[h] < protocol["minimum_fit_rows"] for h in horizons):
        raise RuntimeError("Insufficient matched training observations.")

    sample = pd.concat(sample_parts, ignore_index=True)
    if len(sample) > protocol["maximum_training_sample_rows"]:
        raise RuntimeError("Training diagnostic sample exceeds its row guard.")
    del sample_parts

    fills = {
        h: np.divide(
            sums[h], counts[h], out=np.zeros(width), where=counts[h] > 0
        )
        for h in horizons
    }
    statistics = {h: new_statistics(width * 2) for h in horizons}

    # Second pass: stable centered covariance and cross-product accumulation.
    for _, frame in frames(root, dataset, start, end):
        values = frame[columns].to_numpy(dtype=float)
        for h in horizons:
            _, common = eligible(frame, h, start, end, protocol)
            update_statistics(
                statistics[h],
                transform(values[common.to_numpy()], fills[h]),
                frame.loc[common, f"label_{h}s_pips"].to_numpy(dtype=float),
            )

    training_metrics = sample.loc[sample["regime_metrics_complete"]]
    if len(training_metrics) < protocol["minimum_fit_rows"]:
        raise RuntimeError("Insufficient sampled training liquidity metrics.")
    low = protocol["regime_settings"]["low_quantile"]
    high = protocol["regime_settings"]["high_quantile"]
    thresholds = {
        "rule_version": "training_quantiles_five_minute_sample_v1",
        "settings": protocol["regime_settings"],
        "fit_start_utc": str(start), "fit_end_utc_exclusive": str(end),
        "training_metric_rows": len(training_metrics),
        "cutoffs": {
            "spread_high_pips": float(training_metrics[METRIC_COLUMNS[0]].quantile(high)),
            "volatility_high_pips": float(training_metrics[METRIC_COLUMNS[1]].quantile(high)),
            "depth_low_source_units": float(training_metrics[METRIC_COLUMNS[2]].quantile(low)),
            "activity_low_snapshots": float(training_metrics[METRIC_COLUMNS[3]].quantile(low)),
        },
    }

    result = {
        "training_start": str(start), "training_end_exclusive": str(end),
        "thresholds": thresholds, "horizons": {},
        "maximum_observed_input_frame_bytes": maximum_frame_bytes,
        "training_sample_rows": len(sample),
    }
    price_count = len(settings["price_features"])
    price_indices = list(range(price_count)) + list(
        range(width, width + price_count)
    )
    all_indices = list(range(width * 2))

    for h in horizons:
        fitted = {
            "ridge_price": finish_statistics(
                statistics[h], fills[h], price_indices, columns,
                settings["ridge_alpha"],
            ),
            "ridge_price_size": finish_statistics(
                statistics[h], fills[h], all_indices, columns,
                settings["ridge_alpha"],
            ),
        }
        _, common = eligible(sample, h, start, end, protocol)
        selected = sample.loc[common]
        if len(selected) < 100:
            raise RuntimeError("Too few training observations for forecast buckets.")
        cuts = {}
        for name, model in fitted.items():
            values = predict(model, selected)
            cuts[name] = np.unique(
                np.quantile(values, [.2, .4, .6, .8])
            ).tolist()
        cuts["zero"] = []
        cuts["trailing"] = np.unique(np.quantile(
            selected[f"mid_change_{h}s_pips"], [.2, .4, .6, .8]
        )).tolist()
        result["horizons"][str(h)] = {
            "models": fitted, "bucket_cuts": cuts,
            "training_rows": rows[h],
            "maximum_training_target_utc": maximum_targets[h].isoformat(),
            "all_missing_predictors": [
                name for name, count in zip(columns, counts[h]) if count == 0
            ],
        }
    return result
''')

put_source("src/eurusd_research/completion_evaluation.py", r'''
"""Resumable matched-population evaluation and additive diagnostic totals."""

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from . import features as io
from .completion_data import eligible, frames, save_frame, verify_file
from .completion_models import predict
from .regimes import classify_regimes, METRIC_COLUMNS


MODELS = ["zero", "trailing", "ridge_price", "ridge_price_size"]
SUM_COLUMNS = [
    "rows", "abs_sum", "sq_sum", "actual_sum", "forecast_sum",
    "forecast_sq_sum", "forecast_actual_sum", "spread_sum",
    "abs_forecast_sum", "ratio_sum", "positive_spread_rows",
]


def diagnostic_totals(frame, actual, prediction, dimensions, h, model):
    error = prediction - actual
    spread = frame["spread_pips"].to_numpy(dtype=float)
    positive = spread > 0
    values = pd.DataFrame({
        "rows": np.ones(len(frame), dtype=np.int64),
        "abs_sum": np.abs(error), "sq_sum": error**2,
        "actual_sum": actual, "forecast_sum": prediction,
        "forecast_sq_sum": prediction**2,
        "forecast_actual_sum": prediction * actual,
        "spread_sum": spread,
        "abs_forecast_sum": np.abs(prediction),
        "ratio_sum": np.divide(
            np.abs(prediction), spread,
            out=np.zeros(len(frame)), where=positive,
        ),
        "positive_spread_rows": positive.astype(np.int64),
    })
    outputs = []
    for kind, groups in dimensions.items():
        values["group"] = np.asarray(groups).astype(str)
        grouped = values.groupby("group", sort=False)[SUM_COLUMNS].sum().reset_index()
        grouped["kind"], grouped["horizon_seconds"], grouped["model"] = kind, h, model
        outputs.append(grouped)
    return pd.concat(outputs, ignore_index=True)


def add_totals(accumulator, table):
    # Iterate over small aggregate tables, never over historical quotes.
    for row in table.itertuples(index=False):
        key = (row.kind, row.group, int(row.horizon_seconds), row.model)
        values = np.asarray([getattr(row, name) for name in SUM_COLUMNS], float)
        accumulator[key] = accumulator.get(key, np.zeros(len(SUM_COLUMNS))) + values


def totals_frame(accumulator):
    records = []
    for (kind, group, h, model), values in accumulator.items():
        records.append({
            "kind": kind, "group": group, "horizon_seconds": h, "model": model,
            **dict(zip(SUM_COLUMNS, values.tolist())),
        })
    return pd.DataFrame(records)


def metrics(table):
    out = table.copy()
    n = out["rows"]
    out["mae_pips"] = out["abs_sum"] / n
    out["rmse_pips"] = np.sqrt(out["sq_sum"] / n)
    out["mean_actual_pips"] = out["actual_sum"] / n
    out["mean_forecast_pips"] = out["forecast_sum"] / n
    out["forecast_bias_pips"] = out["mean_forecast_pips"] - out["mean_actual_pips"]
    denominator = out["forecast_sq_sum"] - out["forecast_sum"]**2 / n
    numerator = out["forecast_actual_sum"] - out["forecast_sum"] * out["actual_sum"] / n
    out["calibration_slope"] = numerator / denominator.where(denominator > 1e-15)
    out["calibration_intercept_pips"] = (
        out["mean_actual_pips"] - out["calibration_slope"] * out["mean_forecast_pips"]
    )
    out["mean_spread_pips"] = out["spread_sum"] / n
    out["mean_abs_forecast_pips"] = out["abs_forecast_sum"] / n
    out["mean_abs_forecast_over_positive_spread"] = (
        out["ratio_sum"] / out["positive_spread_rows"].where(
            out["positive_spread_rows"] > 0
        )
    )
    return out


def score_block(frame, fitted, fold, protocol):
    regime_columns = [
        "price_valid", "bid_size", "ask_size", "metric_lookback_seconds",
        "regime_metrics_complete", *METRIC_COLUMNS,
    ]
    regimes = classify_regimes(
        frame[regime_columns], fitted["thresholds"]
    )["liquidity_regime"]
    times = frame["feature_timestamp_utc"]
    sessions = np.select(
        [frame["session_overlap"], frame["london_session"], frame["new_york_session"]],
        ["overlap", "london_only", "new_york_only"], default="other",
    )
    ages = np.select(
        [frame["snapshot_staleness_seconds"].le(1),
         frame["snapshot_staleness_seconds"].le(5)],
        ["0_to_1s", "over_1_to_5s"], default="over_5_to_30s",
    )
    forecast = frame[[
        "feature_timestamp_utc", "source_timestamp_utc", "source_sha256",
        "source_row", "snapshot_staleness_seconds", "spread_pips",
    ]].copy()
    forecast["liquidity_regime"] = regimes
    forecast["session"] = sessions
    keep = np.zeros(len(frame), dtype=bool)
    tables, coverage = [], []

    for h in protocol["horizons_seconds"]:
        price, common = eligible(
            frame, h, fold["evaluation_start"], fold["evaluation_end"], protocol
        )
        mask = common.to_numpy()
        keep |= mask
        forecast[f"eligible_{h}s"] = mask
        coverage.append({
            "horizon_seconds": h,
            "clock_rows": len(frame),
            "price_valid_rows": int(frame["price_valid"].sum()),
            "price_eligible_rows": int(price.sum()),
            "matched_rows": int(common.sum()),
            "price_eligible_without_required_size": int((price & ~common).sum()),
        })
        selected = frame.loc[common]
        info = fitted["horizons"][str(h)]
        actual = selected[f"label_{h}s_pips"].to_numpy(dtype=float)
        predictions = {
            "zero": np.zeros(len(selected)),
            "trailing": selected[f"mid_change_{h}s_pips"].to_numpy(dtype=float),
            **{
                name: predict(model, selected)
                for name, model in info["models"].items()
            },
        }
        for name in [f"actual_{h}s", *[f"{m}_{h}s" for m in MODELS]]:
            forecast[name] = np.nan
        forecast.loc[common, f"actual_{h}s"] = actual

        if not len(selected):
            continue
        dimensions = {
            "date": times.loc[common].dt.strftime("%Y-%m-%d").to_numpy(),
            "year": times.loc[common].dt.year.to_numpy(),
            "session": sessions[mask],
            "regime": regimes.loc[common].to_numpy(),
            "snapshot_age": ages[mask],
            "endpoint": np.where(
                selected[f"same_source_record_{h}s"],
                "same_source_record", "different_source_record",
            ),
        }
        for name, prediction in predictions.items():
            forecast.loc[common, f"{name}_{h}s"] = prediction
            buckets = np.searchsorted(
                info["bucket_cuts"][name], prediction, side="right"
            ) + 1
            tables.append(diagnostic_totals(
                selected, actual, prediction,
                {**dimensions, "bucket": buckets}, h, name,
            ))

    statistics = (
        pd.concat(tables, ignore_index=True)
        if tables else pd.DataFrame(columns=[
            "group", *SUM_COLUMNS, "kind", "horizon_seconds", "model"
        ])
    )
    return forecast.loc[keep].reset_index(drop=True), statistics, coverage


def evaluate_fold(root, dataset, fitted, fold, protocol, run_id):
    root = Path(root)
    directory = root / "artifacts/manifests/completion" / run_id / fold["id"]
    output = root / "artifacts/forecasts/completion" / run_id / fold["id"]
    accumulator, coverage_totals = {}, {}
    committed, reused, forecast_rows, forecast_bytes = 0, 0, 0, 0

    for source_part, frame in frames(
        root, dataset, fold["evaluation_start"], fold["evaluation_end"]
    ):
        if shutil.disk_usage(root).free < protocol["minimum_free_gib_during_run"] * 1024**3:
            raise RuntimeError("Free disk space fell below the configured reserve.")
        key = pd.Timestamp(source_part["start"]).strftime("%Y%m%dT%H%M%S")
        checkpoint = directory / "evaluation_parts" / f"{key}.json"
        if checkpoint.exists():
            saved = json.loads(checkpoint.read_text(encoding="utf-8"))
            if (
                saved["source_sha256"] != source_part["sha256"]
                or saved["fold"] != fold
                or saved["run_id"] != run_id
            ):
                raise RuntimeError("Incompatible evaluation checkpoint.")
            verify_file(root, saved["forecast"])
            stats_path = verify_file(root, saved["statistics"])
            statistics = pd.read_parquet(stats_path)
            reused += 1
        else:
            forecasts, statistics, counts = score_block(frame, fitted, fold, protocol)
            saved = {
                "run_id": run_id, "fold": fold,
                "source_sha256": source_part["sha256"],
                "forecast": save_frame(root, forecasts, output / f"{key}.parquet"),
                "statistics": save_frame(
                    root, statistics, directory / "statistics" / f"{key}.parquet"
                ),
                "coverage": counts,
            }
            io.write_json(checkpoint, saved)
            del forecasts

        add_totals(accumulator, statistics)
        for row in saved["coverage"]:
            h = row["horizon_seconds"]
            total = coverage_totals.setdefault(h, {
                name: 0 for name in row if name != "horizon_seconds"
            })
            for name, value in row.items():
                if name != "horizon_seconds":
                    total[name] += value
        forecast_rows += saved["forecast"]["rows"]
        forecast_bytes += saved["forecast"]["bytes"]
        committed += 1
        if committed % 120 == 0:
            print(f"{fold['id']}: evaluation blocks verified/committed {committed:,}")

    table = totals_frame(accumulator)
    if table.empty:
        raise RuntimeError("Evaluation fold has no eligible observations.")

    for h, counts in coverage_totals.items():
        for model in MODELS:
            observed = table.loc[
                table["kind"].eq("date")
                & table["horizon_seconds"].eq(h)
                & table["model"].eq(model), "rows"
            ].sum()
            if int(observed) != counts["matched_rows"]:
                raise RuntimeError("Forecast population reconciliation failed.")

    records = table.to_dict(orient="records")
    result = {
        "run_id": run_id, "fold": fold, "status": "complete",
        "statistics": records,
        "coverage": [
            {"horizon_seconds": h, **counts}
            for h, counts in coverage_totals.items()
        ],
        "forecast_rows": forecast_rows, "forecast_bytes": forecast_bytes,
        "evaluation_blocks": committed, "evaluation_blocks_reused": reused,
    }
    io.write_json(directory / "evaluation.json", result)
    print(f"Completed {fold['id']}: {committed:,} blocks; {forecast_rows:,} saved forecast rows.")
    return result
''')


put_source("tests/test_completion.py", r'''
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from eurusd_research.completion_models import (
    new_statistics, update_statistics, finish_statistics, predict, transform,
)
from eurusd_research.completion_data import assemble_block, eligible


def test_batched_ridge_matches_sklearn():
    rng = np.random.default_rng(42)
    x = rng.normal(size=(500, 4))
    x[::7, 0] = np.nan
    x[:, 3] = 2.0
    y = rng.normal(size=500)
    fill = np.nanmean(x, axis=0)
    z = transform(x, fill)
    state = new_statistics(z.shape[1])
    for left in range(0, len(y), 37):
        update_statistics(state, z[left:left + 37], y[left:left + 37])
    model = finish_statistics(state, fill, list(range(8)), list("abcd"), 10.0)
    scaler = StandardScaler()
    reference = Ridge(alpha=10.0, solver="cholesky").fit(
        scaler.fit_transform(z), y
    )
    actual = predict(model, pd.DataFrame(x, columns=list("abcd")))
    np.testing.assert_allclose(
        actual, reference.predict(scaler.transform(z)), atol=1e-9, rtol=1e-8
    )


def policy():
    return {
        "feature_settings": {
            "pip_size": .0001, "max_staleness_seconds": 30,
            "substantial_gap_seconds": 30,
            "return_windows_seconds": [1, 5, 30, 60],
            "volatility_windows_seconds": [60, 300],
            "activity_windows_seconds": [1, 5, 30, 60, 300],
            "change_windows_seconds": [1, 5],
            "observed_price_change_window_seconds": 300,
            "wide_spread_review_pips": 20,
            "london_session_start_hour": 8, "london_session_end_hour": 17,
            "new_york_session_start_hour": 8, "new_york_session_end_hour": 17,
            "max_grid_rows_per_block": 100000,
        },
        "regime_settings": {"lookback_seconds": 60},
        "horizons_seconds": [1, 5, 30, 60],
        "maximum_observed_frame_mib": 512,
        "model_settings": {"required_current_size_features": ["bid_size", "ask_size"]},
        "excluded_signal_and_label_intervals": [],
    }


def quotes():
    times = pd.date_range("2020-01-01", periods=1200, freq="s", tz="UTC")
    return pd.DataFrame({
        "timestamp_utc": times,
        "source_row": np.arange(1, len(times) + 1),
        "source_file": "synthetic.csv", "source_sha256": "synthetic",
        "bid": 1.1 + np.sin(np.arange(len(times)) / 17) * .0001,
        "ask": 1.1002 + np.sin(np.arange(len(times)) / 17) * .0001,
        "bid_size": 2.0, "ask_size": 3.0, "source_gap_seconds": 1.0,
        "flag_missing_size": False, "flag_invalid_size": False,
        "flag_zero_size": False, "flag_negative_size": False,
    })


def test_joint_features_labels_partition_consistency():
    q, p = quotes(), policy()
    start = q["timestamp_utc"].iloc[600]
    middle, end = start + pd.Timedelta(seconds=60), start + pd.Timedelta(seconds=120)
    whole, _ = assemble_block(q, start, end, p, q["timestamp_utc"].iloc[0])
    first, _ = assemble_block(q, start, middle, p, q["timestamp_utc"].iloc[0])
    second, _ = assemble_block(q, middle, end, p, q["timestamp_utc"].iloc[0])
    pd.testing.assert_frame_equal(
        whole, pd.concat([first, second], ignore_index=True),
        check_exact=False, atol=1e-9, rtol=1e-9,
    )


def test_future_append_does_not_change_features():
    q, p = quotes(), policy()
    start = q["timestamp_utc"].iloc[600]
    end = start + pd.Timedelta(seconds=120)
    first, _ = assemble_block(q.iloc[:900], start, end, p, q["timestamp_utc"].iloc[0])
    second, _ = assemble_block(q, start, end, p, q["timestamp_utc"].iloc[0])
    pd.testing.assert_frame_equal(first, second)


def test_target_at_evaluation_boundary_is_purged():
    p = policy()
    t = pd.date_range("2020-01-01", periods=3, freq="s", tz="UTC")
    frame = pd.DataFrame({
        "feature_timestamp_utc": t,
        "target_timestamp_utc_1s": t + pd.Timedelta(seconds=1),
        "price_valid": True, "label_valid_1s": True,
        "label_1s_pips": 1.0, "mid_change_1s_pips": 0.0,
        "size_pair_valid": True, "bid_size": 1.0, "ask_size": 1.0,
    })
    _, common = eligible(frame, 1, t[0], t[-1] + pd.Timedelta(seconds=1), p)
    assert common.tolist() == [True, True, False]
''')

test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--color=no"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode:
    raise RuntimeError("Correctness tests failed. Stop and return Cell 12 output.")

TEST_SUMMARY = test_run.stdout.strip().splitlines()[-1]

importlib.invalidate_caches()
from eurusd_research import completion_data as completion_data
from eurusd_research import completion_models as completion_models
from eurusd_research import completion_evaluation as completion_evaluation

completion_data = importlib.reload(completion_data)
completion_models = importlib.reload(completion_models)
completion_evaluation = importlib.reload(completion_evaluation)

import_check = subprocess.run(
    [
        sys.executable, "-c",
        "import eurusd_research.completion_data; "
        "import eurusd_research.completion_models; "
        "import eurusd_research.completion_evaluation; "
        "print('PASS: package imports from notebooks directory')",
    ],
    cwd=REPO_ROOT / "notebooks",
    capture_output=True,
    text=True,
)
print(import_check.stdout)
if import_check.returncode:
    print(import_check.stderr)
    raise RuntimeError("Package import validation failed.")




In [ ]:
implementation_files = sorted(
    (REPO_ROOT / "src/eurusd_research").glob("*.py")
)
PROTOCOL["implementation_sha256"] = {
    path.name: _sha256(path) for path in implementation_files
}

protocol_bytes = json.dumps(
    PROTOCOL, sort_keys=True, allow_nan=False
).encode("utf-8")
RUN_ID = hashlib.sha256(protocol_bytes).hexdigest()[:24]

WORK = REPO_ROOT / "artifacts/manifests/completion" / RUN_ID
TABLES = REPO_ROOT / "reports/tables" / RUN_ID
PRIVATE = REPO_ROOT / "reports/private/completion" / RUN_ID
FIGURES = REPO_ROOT / "reports/figures" / RUN_ID

for directory in (WORK, TABLES, PRIVATE, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)

protocol_path = WORK / "protocol.json"
if protocol_path.exists():
    if json.loads(protocol_path.read_text(encoding="utf-8")) != PROTOCOL:
        raise RuntimeError("Existing run protocol differs.")
else:
    feature_io.write_json(protocol_path, PROTOCOL)

protocol_hash = _sha256(protocol_path)
collection = history.load_collection(REPO_ROOT, history_manifest, verify=True)

boundary = pd.Timestamp(collection["members"][1]["utc_min"]).floor("h")
left = boundary - pd.Timedelta(hours=1)
middle = boundary
right = boundary + pd.Timedelta(hours=1)
index = completion_data.source_index(collection)

quotes = completion_data.read_quotes(
    REPO_ROOT, collection, index,
    left - pd.Timedelta(seconds=REGIME_SETTINGS["lookback_seconds"]),
    right + pd.Timedelta(seconds=60),
    FEATURE_SETTINGS, WORK / "boundary_check",
)

whole, _ = completion_data.assemble_block(
    quotes, left, right, PROTOCOL, collection["scope_utc_min"]
)
first, _ = completion_data.assemble_block(
    quotes, left, middle, PROTOCOL, collection["scope_utc_min"]
)
second, _ = completion_data.assemble_block(
    quotes, middle, right, PROTOCOL, collection["scope_utc_min"]
)

pd.testing.assert_frame_equal(
    whole,
    pd.concat([first, second], ignore_index=True),
    check_exact=False, atol=1e-8, rtol=1e-8,
)

print("PASS: real cross-source continuous-versus-block calculation.")
print("Run ID:", RUN_ID)
print("Protocol checksum:", protocol_hash)
del quotes, whole, first, second

In [ ]:
free_gib = shutil.disk_usage(REPO_ROOT).free / 1024**3
print(f"Available disk space: {free_gib:,.1f} GiB")

if free_gib < 200:
    raise RuntimeError(
        "This consolidated run requires at least 200 GiB free at the start. "
        "Free space before continuing."
    )

dataset_started = time.perf_counter()

from eurusd_research.data import _writer_lock

with _writer_lock(WORK / "dataset_lock"):
    dataset = completion_data.make_dataset(
        REPO_ROOT, collection, PROTOCOL, RUN_ID
    )

print(json.dumps({
    "status": dataset["status"],
    "clock_rows": dataset["rows"],
    "start_utc": dataset["start"],
    "end_utc_exclusive": dataset["end"],
    "parquet_gib": round(dataset["bytes"] / 1024**3, 3),
    "blocks": len(dataset["parts"]),
    "blocks_reused": dataset["blocks_reused"],
    "runtime_seconds_this_execution": dataset["runtime_seconds_this_execution"],
}, indent=2))

In [ ]:
wide_parts = []
ordinal = 0

for part in collection["parts"]:
    path = completion_data.safe_path(REPO_ROOT, part["path"])
    table = pq.ParquetFile(path).read(columns=[
        "timestamp_utc", "flag_extreme_spread", "source_gap_seconds"
    ])
    frame = table.to_pandas()
    mask = frame["flag_extreme_spread"].to_numpy(dtype=bool)
    if mask.any():
        selected = frame.loc[mask, ["timestamp_utc", "source_gap_seconds"]].copy()
        selected["global_record"] = ordinal + np.flatnonzero(mask)
        wide_parts.append(selected)
    ordinal += len(frame)

if wide_parts:
    wide = pd.concat(wide_parts, ignore_index=True)
    new_episode = (
        wide["global_record"].diff().ne(1)
        | wide["source_gap_seconds"].gt(30)
    )
    wide["episode"] = new_episode.cumsum()
    episodes = wide.groupby("episode", sort=True).agg(
        start_utc=("timestamp_utc", "min"),
        last_observed_wide_quote_utc=("timestamp_utc", "max"),
        wide_quote_rows=("global_record", "size"),
    ).reset_index(drop=True)
    episodes["observed_duration_seconds"] = (
        episodes["last_observed_wide_quote_utc"] - episodes["start_utc"]
    ).dt.total_seconds()
    episodes["review_over_20_minutes"] = episodes["observed_duration_seconds"].gt(1200)
else:
    episodes = pd.DataFrame(columns=[
        "start_utc", "last_observed_wide_quote_utc",
        "wide_quote_rows", "observed_duration_seconds",
        "review_over_20_minutes",
    ])

episodes.to_csv(TABLES / "wide_spread_episodes.csv", index=False)
WIDE_SUMMARY = {
    "wide_quote_rows": int(sum(p["wide_quote_rows"] for p in episodes.to_dict("records"))),
    "episodes": len(episodes),
    "episodes_over_20_minutes": int(episodes["review_over_20_minutes"].sum()),
    "maximum_observed_duration_seconds": (
        float(episodes["observed_duration_seconds"].max()) if len(episodes) else None
    ),
    "automatic_exclusions": 0,
}
print(json.dumps(WIDE_SUMMARY, indent=2))

del wide_parts, frame, table

In [ ]:
development_start = pd.Timestamp(PROTOCOL["development_start_utc"])
development_end = pd.Timestamp(PROTOCOL["development_end_utc"])
dataset_start = pd.Timestamp(dataset["start"])

development_folds = []
evaluation_start = development_start

while evaluation_start < development_end:
    evaluation_end = min(
        pd.Timestamp(year=evaluation_start.year + 1, month=1, day=1, tz="UTC"),
        development_end,
    )
    training_start = max(
        dataset_start,
        evaluation_start - pd.Timedelta(days=PROTOCOL["training_calendar_days"]),
    )
    development_folds.append({
        "id": f"development-{evaluation_start.year}",
        "phase": "development",
        "training_start": training_start.isoformat(),
        "training_end": evaluation_start.isoformat(),
        "evaluation_start": evaluation_start.isoformat(),
        "evaluation_end": evaluation_end.isoformat(),
    })
    evaluation_start = evaluation_end

final_start = pd.Timestamp(PROTOCOL["final_test"]["start_utc"])
final_end = pd.Timestamp(PROTOCOL["final_test"]["end_utc_exclusive"])
final_fold = {
    "id": "final-test",
    "phase": "final_test",
    "training_start": (
        final_start - pd.Timedelta(days=PROTOCOL["training_calendar_days"])
    ).isoformat(),
    "training_end": final_start.isoformat(),
    "evaluation_start": final_start.isoformat(),
    "evaluation_end": final_end.isoformat(),
}

pd.DataFrame(development_folds + [final_fold]).to_csv(
    TABLES / "fold_plan.csv", index=False
)
display(pd.DataFrame(development_folds + [final_fold]))

def obtain_fit(fold):
    directory = WORK / fold["id"]
    directory.mkdir(parents=True, exist_ok=True)
    model_path = directory / "models.json"
    metadata_path = directory / "fit_metadata.json"

    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if (
            metadata["protocol_sha256"] != protocol_hash
            or metadata["fold"] != fold
            or _sha256(model_path) != metadata["model_sha256"]
        ):
            raise RuntimeError("Saved model fit is incompatible or changed.")
        fitted = json.loads(model_path.read_text(encoding="utf-8"))
        print("Verified saved fit:", fold["id"])
    else:
        began = time.perf_counter()
        fitted = completion_models.fit_fold(
            REPO_ROOT, dataset,
            fold["training_start"], fold["training_end"], PROTOCOL,
        )
        feature_io.write_json(model_path, fitted)
        feature_io.write_json(metadata_path, {
            "protocol_sha256": protocol_hash,
            "fold": fold,
            "model_sha256": _sha256(model_path),
            "fit_seconds": round(time.perf_counter() - began, 3),
        })
        print("Fitted:", fold["id"])

    for h, information in fitted["horizons"].items():
        if pd.Timestamp(information["maximum_training_target_utc"]) >= pd.Timestamp(
            fold["evaluation_start"]
        ):
            raise RuntimeError("Training targets reach the evaluation period.")
    return fitted

development_results = []
development_started = time.perf_counter()

with _writer_lock(WORK / "evaluation_lock"):
    for fold in development_folds:
        fitted = obtain_fit(fold)
        result = completion_evaluation.evaluate_fold(
            REPO_ROOT, dataset, fitted, fold, PROTOCOL, RUN_ID
        )
        development_results.append(result)

development_seconds = time.perf_counter() - development_started
print("Development folds completed:", len(development_results))
print("Final test evaluated: False")

In [ ]:
final_fitted = obtain_fit(final_fold)

FROZEN = {
    "run_id": RUN_ID,
    "protocol_sha256": protocol_hash,
    "final_fold": final_fold,
    "final_model_sha256": _sha256(WORK / "final-test/models.json"),
    "source_manifest_sha256": _sha256(history_manifest),
    "selection": "Fixed protocol; no tuning or feature search in Notebook 09",
    "models": completion_evaluation.MODELS,
    "horizons_seconds": PROTOCOL["horizons_seconds"],
    "h001": "Not included; remains untested future work",
}

freeze_path = WORK / "final_test_freeze.json"
if freeze_path.exists():
    if json.loads(freeze_path.read_text(encoding="utf-8")) != FROZEN:
        raise RuntimeError("Existing final-test freeze differs. Stop.")
else:
    feature_io.write_json(freeze_path, FROZEN)

freeze_hash = _sha256(freeze_path)
print("FINAL-TEST PROTOCOL FROZEN")
print(json.dumps(FROZEN, indent=2))

In [ ]:
if _sha256(protocol_path) != protocol_hash:
    raise RuntimeError("Protocol changed after freezing.")
if _sha256(freeze_path) != freeze_hash:
    raise RuntimeError("Final-test freeze changed.")
if _sha256(WORK / "final-test/models.json") != FROZEN["final_model_sha256"]:
    raise RuntimeError("Final model changed after freezing.")

final_started = time.perf_counter()

with _writer_lock(WORK / "evaluation_lock"):
    final_result = completion_evaluation.evaluate_fold(
        REPO_ROOT, dataset, final_fitted, final_fold, PROTOCOL, RUN_ID
    )

final_seconds = time.perf_counter() - final_started
print("PASS: frozen final-test evaluation completed.")
print("Final-test dates:", final_fold["evaluation_start"], "to", final_fold["evaluation_end"])

In [ ]:
from eurusd_research.walk_forward import paired_block_interval

all_results = development_results + [final_result]
raw_tables = []
coverage_tables = []

for result in all_results:
    phase = result["fold"]["phase"]
    fold_id = result["fold"]["id"]

    raw = pd.DataFrame(result["statistics"])
    raw["phase"], raw["fold"] = phase, fold_id
    raw_tables.append(raw)

    coverage = pd.DataFrame(result["coverage"])
    coverage["phase"], coverage["fold"] = phase, fold_id
    coverage_tables.append(coverage)

raw_statistics = pd.concat(raw_tables, ignore_index=True)
coverage = pd.concat(coverage_tables, ignore_index=True)
coverage.to_csv(TABLES / "coverage.csv", index=False)

SUM_COLUMNS = completion_evaluation.SUM_COLUMNS
daily = raw_statistics.loc[raw_statistics["kind"].eq("date")].copy()

overall_sums = daily.groupby(
    ["phase", "horizon_seconds", "model"], as_index=False
)[SUM_COLUMNS].sum()

overall = completion_evaluation.metrics(overall_sums)

zero_reference = overall.loc[
    overall["model"].eq("zero"),
    ["phase", "horizon_seconds", "mae_pips", "rmse_pips"],
].rename(columns={
    "mae_pips": "zero_mae_pips",
    "rmse_pips": "zero_rmse_pips",
})

overall = overall.merge(
    zero_reference,
    on=["phase", "horizon_seconds"],
    validate="many_to_one",
)
overall["mae_gain_vs_zero_pips"] = overall["zero_mae_pips"] - overall["mae_pips"]
overall["rmse_gain_vs_zero_pips"] = overall["zero_rmse_pips"] - overall["rmse_pips"]

overall.to_csv(TABLES / "overall_metrics.csv", index=False)

for kind in ["date", "year", "session", "regime", "snapshot_age", "endpoint", "bucket"]:
    subset = raw_statistics.loc[raw_statistics["kind"].eq(kind)]
    grouped = subset.groupby(
        ["phase", "horizon_seconds", "model", "group"], as_index=False
    )[SUM_COLUMNS].sum()
    completion_evaluation.metrics(grouped).to_csv(
        TABLES / f"by_{kind}.csv", index=False
    )

by_fold = daily.groupby(
    ["phase", "fold", "horizon_seconds", "model"], as_index=False
)[SUM_COLUMNS].sum()
completion_evaluation.metrics(by_fold).to_csv(
    TABLES / "by_fold.csv", index=False
)

interval_records = []
comparisons = [
    ("ridge_price", "zero"),
    ("ridge_price_size", "ridge_price"),
    ("ridge_price_size", "zero"),
]

for phase in ["development", "final_test"]:
    for h in PROTOCOL["horizons_seconds"]:
        selected = daily.loc[
            daily["phase"].eq(phase) & daily["horizon_seconds"].eq(h)
        ]
        for candidate, reference in comparisons:
            left = selected.loc[selected["model"].eq(candidate), [
                "group", "rows", "abs_sum", "sq_sum"
            ]].rename(columns={
                "group": "date_utc",
                "abs_sum": "candidate_abs_sum",
                "sq_sum": "candidate_sq_sum",
            })
            right = selected.loc[selected["model"].eq(reference), [
                "group", "rows", "abs_sum", "sq_sum"
            ]].rename(columns={
                "group": "date_utc",
                "rows": "reference_rows",
                "abs_sum": "reference_abs_sum",
                "sq_sum": "reference_sq_sum",
            })
            paired = left.merge(right, on="date_utc", validate="one_to_one")
            if not paired["rows"].equals(paired["reference_rows"]):
                raise RuntimeError("Paired uncertainty populations differ.")

            for block in PROTOCOL["bootstrap_block_dates"]:
                estimate = paired_block_interval(
                    paired,
                    block_dates=block,
                    repetitions=PROTOCOL["bootstrap_repetitions"],
                    confidence=PROTOCOL["confidence"],
                    seed=PROTOCOL["random_seed"],
                    minimum_dates=30,
                )
                interval_records.append({
                    "phase": phase, "horizon_seconds": h,
                    "candidate": candidate, "reference": reference,
                    **estimate,
                })

intervals = pd.DataFrame(interval_records)
intervals.to_csv(TABLES / "paired_block_intervals.csv", index=False)

size_comparison = overall.loc[
    overall["model"].isin(["ridge_price", "ridge_price_size"])
].pivot(
    index=["phase", "horizon_seconds"],
    columns="model",
    values=["mae_pips", "rmse_pips"],
)
size_comparison = pd.DataFrame({
    "delta_mae_size_minus_price_pips": (
        size_comparison["mae_pips"]["ridge_price_size"]
        - size_comparison["mae_pips"]["ridge_price"]
    ),
    "delta_rmse_size_minus_price_pips": (
        size_comparison["rmse_pips"]["ridge_price_size"]
        - size_comparison["rmse_pips"]["ridge_price"]
    ),
}).reset_index()
size_comparison.to_csv(TABLES / "displayed_size_comparison.csv", index=False)

display(overall[[
    "phase", "horizon_seconds", "model", "rows",
    "mae_pips", "rmse_pips",
    "mae_gain_vs_zero_pips", "rmse_gain_vs_zero_pips",
]])
display(size_comparison)

In [ ]:
final_metrics = overall.loc[overall["phase"].eq("final_test")].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric in zip(axes, ["mae_pips", "rmse_pips"]):
    for model in completion_evaluation.MODELS:
        subset = final_metrics.loc[final_metrics["model"].eq(model)].sort_values(
            "horizon_seconds"
        )
        axis.plot(
            subset["horizon_seconds"].astype(str),
            subset[metric],
            marker="o", label=model,
        )
    axis.set_xlabel("Forecast horizon, seconds")
    axis.set_ylabel(metric.replace("_", " "))
    axis.grid(alpha=.25)
axes[0].legend(fontsize=8)
fig.suptitle("Frozen final-test forecast errors")
fig.tight_layout()
fig.savefig(FIGURES / "final_test_errors.png", dpi=160)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric in zip(axes, [
    "delta_mae_size_minus_price_pips",
    "delta_rmse_size_minus_price_pips",
]):
    for phase in ["development", "final_test"]:
        subset = size_comparison.loc[
            size_comparison["phase"].eq(phase)
        ].sort_values("horizon_seconds")
        axis.plot(
            subset["horizon_seconds"].astype(str),
            subset[metric], marker="o", label=phase,
        )
    axis.axhline(0, color="black", linewidth=1)
    axis.set_xlabel("Forecast horizon, seconds")
    axis.set_ylabel("Size model error minus price model error, pips")
    axis.set_title("MAE" if "mae" in metric else "RMSE")
    axis.grid(alpha=.25)
axes[0].legend()
fig.suptitle("Displayed-size contribution: below zero means lower error")
fig.tight_layout()
fig.savefig(FIGURES / "displayed_size_contribution.png", dpi=160)
plt.show()

yearly = pd.read_csv(TABLES / "by_year.csv")
fig, axis = plt.subplots(figsize=(10, 4))
for model in ["zero", "ridge_price", "ridge_price_size"]:
    subset = yearly.loc[
        yearly["horizon_seconds"].eq(5)
        & yearly["model"].eq(model)
        & yearly["phase"].eq("development")
    ].sort_values("group")
    axis.plot(subset["group"].astype(str), subset["rmse_pips"], marker="o", label=model)
axis.set_xlabel("Development evaluation year")
axis.set_ylabel("5-second RMSE, pips")
axis.set_title("Annual development stability")
axis.grid(alpha=.25)
axis.legend()
fig.tight_layout()
fig.savefig(FIGURES / "annual_stability.png", dpi=160)
plt.show()

annual_dataset = pd.DataFrame([{
    "year": pd.Timestamp(part["start"]).year,
    "clock_rows": part["rows"],
    "price_valid_rows": part["price_valid_rows"],
    "size_valid_rows": part["size_valid_rows"],
    **{
        f"label_valid_{h}s": part["label_valid_rows"][str(h)]
        for h in PROTOCOL["horizons_seconds"]
    },
} for part in dataset["parts"]]).groupby("year", as_index=False).sum()

annual_dataset["calendar_price_coverage_pct"] = (
    100 * annual_dataset["price_valid_rows"] / annual_dataset["clock_rows"]
)
annual_dataset.to_csv(TABLES / "annual_dataset_coverage.csv", index=False)
display(annual_dataset)

total_coverage = coverage.groupby(
    ["phase", "horizon_seconds"], as_index=False
)[[
    "clock_rows", "price_valid_rows", "price_eligible_rows",
    "matched_rows", "price_eligible_without_required_size",
]].sum()
total_coverage["matched_pct_of_usable_prices"] = (
    100 * total_coverage["matched_rows"] / total_coverage["price_valid_rows"]
)
total_coverage.to_csv(TABLES / "evaluation_coverage.csv", index=False)
display(total_coverage)

In [ ]:
def markdown_table(frame, columns, digits=6):
    selected = frame[columns]
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for values in selected.itertuples(index=False, name=None):
        cells = []
        for value in values:
            if pd.isna(value):
                cells.append("—")
            elif isinstance(value, (float, np.floating)):
                cells.append(f"{value:.{digits}f}")
            else:
                cells.append(str(value))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

def update_generated_section(relative, title, body):
    path = REPO_ROOT / relative
    start_marker = "<!-- COMPLETION_RESULTS_START -->"
    end_marker = "<!-- COMPLETION_RESULTS_END -->"
    original = path.read_text(encoding="utf-8") if path.exists() else f"# {title}\n"

    if start_marker in original or end_marker in original:
        if original.count(start_marker) != 1 or original.count(end_marker) != 1:
            raise RuntimeError(f"Ambiguous generated documentation markers: {relative}")
        before, remainder = original.split(start_marker, 1)
        _, after = remainder.split(end_marker, 1)
    else:
        before, after = original.rstrip() + "\n\n", "\n"

    before = before.replace(
        "Results pending execution and validation.",
        "Measured results are recorded in the generated completion section below.",
    )
    content = before + start_marker + "\n" + body.strip() + "\n" + end_marker + after
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(content)

metric_columns = [
    "horizon_seconds", "model", "rows", "mae_pips", "rmse_pips",
    "mae_gain_vs_zero_pips", "rmse_gain_vs_zero_pips",
]
final_table = markdown_table(final_metrics, metric_columns)
size_table = markdown_table(
    size_comparison.loc[size_comparison["phase"].eq("final_test")],
    ["horizon_seconds", "delta_mae_size_minus_price_pips",
     "delta_rmse_size_minus_price_pips"],
)

winners = []
for h, group in final_metrics.groupby("horizon_seconds"):
    best_mae = group.loc[group["mae_pips"].idxmin(), "model"]
    best_rmse = group.loc[group["rmse_pips"].idxmin(), "model"]
    winners.append(
        f"- {h}-second horizon: lowest MAE = `{best_mae}`; "
        f"lowest RMSE = `{best_rmse}`."
    )

size_rows = size_comparison.loc[size_comparison["phase"].eq("final_test")]
mae_improvements = int(size_rows["delta_mae_size_minus_price_pips"].lt(0).sum())
rmse_improvements = int(size_rows["delta_rmse_size_minus_price_pips"].lt(0).sum())

findings = (
    "\n".join(winners)
    + f"\n- Adding displayed-size predictors reduced final-test MAE at "
      f"{mae_improvements}/4 horizons and RMSE at {rmse_improvements}/4 horizons."
    + "\n- Point-estimate rankings must be read alongside the paired block intervals."
)

limitations = """
- This is a research prototype using one Dukascopy/JForex quote feed.
- Displayed sizes are not executed trade volumes; their export denomination remains unresolved.
- Europe/Helsinki timestamp interpretation retains its documented verification limits.
- True side-specific quote ages are unavailable.
- Thirty-second snapshot eligibility is a research assumption; held-snapshot diagnostics are reported.
- Annual fixed-window fits do not establish performance under alternative refit policies.
- Full-history fitting uses mean imputation; pilot fitting used median imputation.
- Liquidity thresholds and forecast buckets use a deterministic five-minute training sample.
- Bootstrap intervals are conditional on fitted forecasts and may not capture all dependence.
- Subgroup and multiple-model comparisons are descriptive.
- H001 remains untested and is not a validated finding.
- No execution, costs, queue position, fill probability, inventory policy, or market-making P&L is modeled.
- Predictive accuracy does not establish achievable trading profits.
"""

disclosure = (
    "Development approach: This is an AI-assisted research project.\n"
    "Raymond Fendi defined the business objectives and provided FX domain\n"
    "direction. OpenAI Codex assisted with implementation, testing, and\n"
    "documentation. CONTRIBUTIONS.md records completed design decisions,\n"
    "reviews, and validation."
)

results_body = f"""
## Completed research run

Run ID: `{RUN_ID}`.

Development evaluation: 2017-01-01 through 2025-10-01, end exclusive.
Final test: 2025-10-01 through 2026-10-01, end exclusive.

Training: up to 365 preceding calendar days; annual development refits.
Final-test models were frozen before scoring and held fixed during the final test.

### Measured final-test results

{final_table}

### Displayed-size comparison

Positive differences mean the size model has higher error.

{size_table}

### Measured findings

{findings}

### Interpretation and pricing relevance

The results measure forecast errors and conditional liquidity states.
Any reliable improvement would motivate further pricing research.
Weak, unstable, or negative improvements do not support adding forecast-driven
pricing adjustments on the evidence of this experiment.

The regime tables describe conditions under transparent training-derived rules.
They do not establish that a stressed label is a data error or a causal predictor.

Forecast magnitudes are compared with prevailing spreads in the saved tables.
This comparison does not establish executable returns.

### Validation

- Project tests: {TEST_SUMMARY}.
- Real cross-source continuous-versus-block calculation passed.
- Canonical source integrity and dataset row conservation passed.
- Training targets were purged at every evaluation boundary.
- Model comparisons used matched observations.
- Final-test protocol and model checksums were frozen before scoring.
- Owner review of Notebook 09 remains pending.

### Wide-spread investigation

Qualifying raw wide quotes: {WIDE_SUMMARY["wide_quote_rows"]}.
Observed episodes: {WIDE_SUMMARY["episodes"]}.
Episodes exceeding 20 minutes: {WIDE_SUMMARY["episodes_over_20_minutes"]}.
Automatic exclusions: zero.

Duration is measured conservatively between observed qualifying quotes and
requires no intervening non-wide record or gap above 30 seconds.

### Limitations

{limitations}

### Future work

H001, alternative training/refit policies, stronger source-timezone confirmation,
and executable-price diagnostics are future experiments. None is represented
as completed here.
"""

update_generated_section(
    "reports/research_results.md", "Research results", results_body
)
update_generated_section(
    "reports/data_quality.md", "Data quality",
    f"""## Completion-stage quality checks

Canonical quotes: {collection["rows_in_scope"]:,}.
Research clock rows: {dataset["rows"]:,}.
Research range: {dataset["start"]} to {dataset["end"]}, end exclusive.

Cross-source context and continuous-versus-block checks passed.
Wide-spread investigation: {json.dumps(WIDE_SUMMARY)}.

Wide spreads, locked quotes, and zero-size observations remain in canonical data.
Feature and label eligibility is separate from raw quality flags.
Coverage tables report the resulting populations.

Timestamp and displayed-size limitations remain documented in the research results.
""",
)

readme_body = f"""
## First research version — executed results

{disclosure}

### Purpose and business relevance

Investigate short-horizon EURUSD mid-price forecasting and liquidity conditions
using Level 1 quotes and displayed sizes, for electronic FX pricing research.

### Data and units

EURUSD prices are USD per EUR; one pip is 0.0001.
Quotes are combined snapshots, converted to UTC using the documented source-timezone assumption.
Displayed-size denomination remains unresolved.

### Repository structure and execution

- `notebooks/`: sequential research explanations and execution.
- `src/eurusd_research/`: reusable calculations.
- `tests/`: financial and temporal correctness checks.
- `configs/`: safe example configuration and ignored machine-local configuration.
- `data/`: original and generated data; large datasets remain ignored.
- `artifacts/`: ignored manifests, models, and forecasts.
- `reports/`: public aggregate documentation and ignored detailed outputs.

Run notebooks in numerical order. Notebook 09 completes scaling, final evaluation,
documentation, and presentation material. It requires completed Notebook 08 outputs.

Installation uses the editable project and its `modeling`, `test`, and
`notebooks` optional dependencies in the active Jupyter environment.
No dependency versions were automatically upgraded by Notebook 09.

### Methodology and validation

One-second backward-looking sampling, 30-second maximum snapshot age,
causal features, exact-horizon labels, chronological purged evaluation,
training-only transformations, matched feature-family comparisons,
and paired date-block uncertainty estimates.

{TEST_SUMMARY}. Detailed methods and limitations are in
`reports/research_results.md`.

### Results

Run `{RUN_ID}` completed development and the reserved 12-month final test.

{findings}

### Observed information and limitations

{limitations}

### Next steps

Review the measured findings and presentation brief.
Further experiments are optional subsequent research and require new development protocols.
"""
update_generated_section("README.md", "EURUSD Fair-Value and Liquidity Research", readme_body)

update_generated_section(
    "CONTRIBUTIONS.md", "Contributions",
    f"""## Completion run `{RUN_ID}`

Raymond Fendi defined the objectives, supplied FX domain direction, selected
the 30-second snapshot policy, retained one-second-return volatility, and
explained the cross-file gap checkpoint.

OpenAI Codex provided the Notebook 09 implementation and documentation.

Recorded execution evidence:
- {TEST_SUMMARY}.
- Full research dataset generated with bounded processing.
- Annual development evaluation and frozen final test executed.
- Matched comparisons and uncertainty tables generated.
- Wide-spread episode investigation completed.

This execution record is based on automated notebook checks.
Owner review, interpretation, and approval of Notebook 09 remain pending.

H001 remains recorded and untested.
""",
)

# Safe aggregate results suitable for repository review.
overall[[
    "phase", "horizon_seconds", "model", "rows",
    "mae_pips", "rmse_pips",
    "mae_gain_vs_zero_pips", "rmse_gain_vs_zero_pips",
]].to_csv(REPO_ROOT / "reports/final_aggregate_metrics.csv", index=False)

presentation = f"""# EURUSD Fair-Value and Liquidity Research — presentation brief

## Slide 1 — Research question
Can Level 1 quotes and displayed sizes improve short-horizon mid-price forecasts
and describe deteriorating liquidity conditions?

Audience: electronic FX market-making research.
Scope: research prototype; no profitability claim.

## Slide 2 — Dataset
- {collection["rows_in_scope"]:,} canonical EURUSD quote records.
- Source coverage: {collection["scope_utc_min"]} to {collection["scope_utc_max"]}.
- Complete-month research evaluation ends October 1, 2026.
- One-second sampling; maximum snapshot age 30 seconds.
- Single-feed limitations and unresolved displayed-size denomination.

## Slide 3 — Leakage controls
- Backward-looking quotes and causal features.
- Exact-horizon targets and gap rejection.
- Training-only preprocessing and regime thresholds.
- Purged training targets.
- Matched model comparison populations.
- Frozen 12-month final test.

## Slide 4 — Models
- Zero-change benchmark.
- Trailing-return benchmark.
- Price-only Ridge.
- Price-plus-size Ridge.
- Up to 365 days of training; no parameter search.
- Mean imputation supports bounded full-history fitting.

## Slide 5 — Measured final-test results
{final_table}

## Slide 6 — Does displayed size help?
{size_table}

{findings}

Use `displayed_size_contribution.png` and the paired block interval table.
Discuss economic magnitude and stability, not only the sign of an error difference.

## Slide 7 — Liquidity and stability
Use:
- `annual_stability.png`;
- `by_regime.csv`;
- `by_session.csv`;
- `by_snapshot_age.csv`;
- `annual_dataset_coverage.csv`.

Training-derived regime labels describe relative conditions.
They are not universal labels or automatic exclusions.

## Slide 8 — Pricing implications and limitations
Forecast performance alone does not justify a live pricing adjustment.
Any proposed adjustment would need a separate execution, latency, cost,
liquidity, inventory, and risk assessment.

{limitations}

## Slide 9 — Conclusion
Present the measured ranking and uncertainty, including negative findings.
Distinguish the evidence from interpretation.

H001 remains an optional future hypothesis, not a completed finding.

## Slide 10 — Development and validation
{disclosure}

Automated validation: {TEST_SUMMARY}.
Owner review of the final notebook remains pending.
"""

presentation_path = REPO_ROOT / "reports/project_presentation_brief.md"
with presentation_path.open("w", encoding="utf-8", newline="\n") as handle:
    handle.write(presentation)

print("Documentation updated:")
for relative in [
    "README.md", "CONTRIBUTIONS.md",
    "reports/research_results.md", "reports/data_quality.md",
    "reports/final_aggregate_metrics.csv",
    "reports/project_presentation_brief.md",
]:
    print(relative)

In [ ]:
if _sha256(history_manifest) != PROTOCOL["source_manifest_sha256"]:
    raise RuntimeError("Canonical history manifest changed during execution.")
if _sha256(protocol_path) != protocol_hash or _sha256(freeze_path) != freeze_hash:
    raise RuntimeError("Frozen research protocol changed.")

for path in implementation_files:
    if _sha256(path) != PROTOCOL["implementation_sha256"][path.name]:
        raise RuntimeError(f"Implementation changed during execution: {path.name}")

if dataset["rows"] != sum(part["rows"] for part in dataset["parts"]):
    raise RuntimeError("Final dataset row conservation failed.")

for phase in ["development", "final_test"]:
    for h in PROTOCOL["horizons_seconds"]:
        selected = overall.loc[
            overall["phase"].eq(phase)
            & overall["horizon_seconds"].eq(h)
        ]
        if len(selected) != 4 or selected["rows"].nunique() != 1:
            raise RuntimeError("Final matched-model reconciliation failed.")
        if not np.isfinite(selected[["mae_pips", "rmse_pips"]]).all().all():
            raise RuntimeError("Nonfinite final evaluation metrics.")

def clean_records(frame):
    return json.loads(frame.to_json(orient="records"))

summary = {
    "notebook": "09_complete_research_and_reporting.ipynb",
    "run_id": RUN_ID,
    "status": "automated completion checks passed; owner review pending",
    "python_version": platform.python_version(),
    "package_versions": VERSIONS,
    "configuration": {
        "history_collection_id": collection["job_id"],
        "training_calendar_days": PROTOCOL["training_calendar_days"],
        "development_folds": len(development_folds),
        "development_start_utc": PROTOCOL["development_start_utc"],
        "development_end_utc_exclusive": PROTOCOL["development_end_utc"],
        "final_test": PROTOCOL["final_test"],
        "maximum_snapshot_age_seconds": 30,
        "volatility_return_interval_seconds": 1,
        "ridge_alpha": 10.0,
        "imputation": PROTOCOL["imputation"],
        "threshold_sample_seconds": 300,
        "ridge_training_population": "all eligible matched one-second training rows",
        "parameter_search": "none",
    },
    "inputs": {
        "canonical_quote_rows": collection["rows_in_scope"],
        "first_quote_utc": collection["scope_utc_min"],
        "last_quote_utc": collection["scope_utc_max"],
    },
    "dataset": {
        "clock_rows": dataset["rows"],
        "start_utc": dataset["start"],
        "end_utc_exclusive": dataset["end"],
        "parquet_bytes": dataset["bytes"],
        "blocks": len(dataset["parts"]),
        "blocks_reused": dataset["blocks_reused"],
        "price_valid_rows": sum(p["price_valid_rows"] for p in dataset["parts"]),
        "size_valid_rows": sum(p["size_valid_rows"] for p in dataset["parts"]),
        "label_valid_rows": {
            str(h): sum(p["label_valid_rows"][str(h)] for p in dataset["parts"])
            for h in PROTOCOL["horizons_seconds"]
        },
    },
    "final_test_metrics": clean_records(final_metrics[[
        "horizon_seconds", "model", "rows", "mae_pips", "rmse_pips",
        "mae_gain_vs_zero_pips", "rmse_gain_vs_zero_pips",
        "calibration_slope", "mean_spread_pips",
        "mean_abs_forecast_pips",
    ]]),
    "displayed_size_comparison": clean_records(size_comparison),
    "wide_spread_investigation": WIDE_SUMMARY,
    "validation": {
        "pytest": TEST_SUMMARY,
        "package_imports_from_notebooks_directory": "passed",
        "real_cross_source_partition_consistency": "passed",
        "dataset_row_conservation": "passed",
        "training_target_boundary_purging": "checked",
        "matched_model_populations": "passed",
        "final_test_protocol_frozen": True,
        "final_test_predictively_evaluated": True,
        "source_manifest_unchanged": True,
        "implementation_unchanged": True,
        "owner_review": "pending",
    },
    "measurements": {
        "dataset_seconds_this_execution": dataset["runtime_seconds_this_execution"],
        "development_seconds_this_execution": round(development_seconds, 3),
        "final_test_seconds_this_execution": round(final_seconds, 3),
        "forecast_parquet_bytes": sum(r["forecast_bytes"] for r in all_results),
        "maximum_observed_dataset_frame_bytes": max(
            p["observed_frame_bytes"] for p in dataset["parts"]
        ),
        "notebook_elapsed_seconds": round(time.perf_counter() - started_notebook, 3),
    },
    "warnings": [
        "Mean imputation differs from the earlier median-imputed pilot.",
        "Threshold and forecast-bucket quantiles use a training-only five-minute sample.",
        "Calendar coverage includes closed periods.",
        "Displayed-size denomination and true side-specific quote ages remain unresolved.",
        "Timezone interpretation retains its documented verification limits.",
        "Bootstrap intervals are conditional on forecasts and comparisons are descriptive.",
        "H001 remains untested future work.",
        "Frame-memory measurements are not total process peak memory.",
        "No executable strategy or achievable-profit conclusion is established.",
    ],
    "saved_outputs": {
        "protocol": str(protocol_path.relative_to(REPO_ROOT)),
        "dataset_manifest": str((WORK / "dataset.json").relative_to(REPO_ROOT)),
        "final_test_freeze": str(freeze_path.relative_to(REPO_ROOT)),
        "tables_directory": str(TABLES.relative_to(REPO_ROOT)),
        "figures_directory": str(FIGURES.relative_to(REPO_ROOT)),
        "forecasts_directory": f"artifacts/forecasts/completion/{RUN_ID}",
        "results": "reports/research_results.md",
        "presentation_brief": "reports/project_presentation_brief.md",
        "public_aggregate_metrics": "reports/final_aggregate_metrics.csv",
        "execution_summary": "reports/private/notebook09_execution.json",
    },
}

feature_io.write_json(
    REPO_ROOT / "reports/private/notebook09_execution.json", summary
)
feature_io.write_json(WORK / "completion.json", summary)

print("NOTEBOOK_09_SUMMARY")
print(json.dumps(summary, indent=2, allow_nan=False))

In [18]:
from pathlib import Path
import hashlib
import json

import duckdb
import pandas as pd
from IPython.display import display

RUN = "66d2e5a9ff903fe48113c6ea"

# Works after a kernel restart.
candidates = [Path.cwd(), *Path.cwd().parents]
root = next(
    (
        p.resolve() for p in candidates
        if (p / f"artifacts/manifests/completion/{RUN}/final-test/evaluation.json").is_file()
    ),
    None,
)
if root is None:
    root = Path(
        input("Full repository folder path: ").strip().strip('"')
    ).expanduser().resolve()

run_directory = root / "artifacts/manifests/completion" / RUN
evaluation = json.loads(
    (run_directory / "final-test/evaluation.json").read_text(encoding="utf-8")
)
if evaluation["status"] != "complete":
    raise RuntimeError("The final-test evaluation is incomplete.")

expected_rows = next(
    row["matched_rows"]
    for row in evaluation["coverage"]
    if row["horizon_seconds"] == 30
)

# Use committed forecast files only; verify their saved checksums.
files = []
checkpoints = sorted(
    (run_directory / "final-test/evaluation_parts").glob("*.json")
)
if not checkpoints:
    raise RuntimeError("No committed final-test forecast blocks found.")

for checkpoint in checkpoints:
    saved = json.loads(checkpoint.read_text(encoding="utf-8"))
    if saved["run_id"] != RUN or saved["fold"] != evaluation["fold"]:
        raise RuntimeError("Forecast checkpoint belongs to another evaluation.")

    record = saved["forecast"]
    path = (root / record["path"]).resolve()
    if not path.is_relative_to(root):
        raise RuntimeError("Forecast path escapes the repository.")

    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    if path.stat().st_size != record["bytes"] or digest.hexdigest() != record["sha256"]:
        raise RuntimeError(f"Forecast integrity failed: {path.name}")
    files.append(str(path))

connection = duckdb.connect(config={"memory_limit": "256MB", "threads": 2})
try:
    connection.read_parquet(
        files, hive_partitioning=False
    ).create_view("saved_forecasts")

    # Every comparison uses the existing matched 30-second population.
    connection.execute("""
        CREATE TEMP VIEW comparison AS
        SELECT
            feature_timestamp_utc,
            liquidity_regime,
            actual_30s,
            zero_30s,
            trailing_30s,
            ridge_price_30s,
            ridge_price_size_30s
        FROM saved_forecasts
        WHERE eligible_30s
    """)

    checks = connection.execute("""
        SELECT
            COUNT(*) AS rows,
            COUNT(*) FILTER (
                WHERE isfinite(actual_30s) IS NOT TRUE
                   OR isfinite(zero_30s) IS NOT TRUE
                   OR isfinite(trailing_30s) IS NOT TRUE
                   OR isfinite(ridge_price_30s) IS NOT TRUE
                   OR isfinite(ridge_price_size_30s) IS NOT TRUE
                   OR liquidity_regime IS NULL
                   OR liquidity_regime NOT IN ('normal', 'thin', 'stressed', 'unclassified')
            ) AS invalid_rows,
            MAX(ABS(zero_30s)) AS maximum_zero_forecast,
            MIN(feature_timestamp_utc) AS first_timestamp,
            MAX(feature_timestamp_utc) AS last_timestamp
        FROM comparison
    """).fetchdf().iloc[0]

    if int(checks["rows"]) != expected_rows or int(checks["invalid_rows"]) != 0:
        raise RuntimeError("Population reconciliation or forecast validity failed.")
    if checks["maximum_zero_forecast"] != 0:
        raise RuntimeError("Zero-change forecasts are unexpectedly nonzero.")

    start = pd.Timestamp(evaluation["fold"]["evaluation_start"])
    end = pd.Timestamp(evaluation["fold"]["evaluation_end"])
    if not (
        start <= pd.Timestamp(checks["first_timestamp"])
        <= pd.Timestamp(checks["last_timestamp"]) < end
    ):
        raise RuntimeError("Forecast dates fall outside the final-test period.")

    # Include errors exactly equal to one pip.
    # The tiny tolerance handles floating-point arithmetic only.
    limit = 1.0 + 1e-9

    counts = connection.execute(f"""
        WITH hits AS (
            SELECT
                liquidity_regime,
                ABS(actual_30s - zero_30s) <= {limit} AS zero_hit,
                ABS(actual_30s - trailing_30s) <= {limit} AS trailing_hit,
                ABS(actual_30s - ridge_price_30s) <= {limit} AS price_hit,
                ABS(actual_30s - ridge_price_size_30s) <= {limit} AS size_hit
            FROM comparison
        )
        SELECT
            CASE WHEN GROUPING(liquidity_regime) = 1
                 THEN 'ALL' ELSE liquidity_regime END AS regime,
            COUNT(*) AS rows,
            COUNT(*) FILTER (WHERE zero_hit) AS zero_hits,
            COUNT(*) FILTER (WHERE trailing_hit) AS trailing_hits,
            COUNT(*) FILTER (WHERE price_hit) AS price_hits,
            COUNT(*) FILTER (WHERE size_hit) AS size_hits,
            COUNT(*) FILTER (WHERE size_hit AND NOT zero_hit) AS size_only_hits,
            COUNT(*) FILTER (WHERE zero_hit AND NOT size_hit) AS zero_only_hits
        FROM hits
        GROUP BY GROUPING SETS ((), (liquidity_regime))
    """).fetchdf()
finally:
    connection.close()

for model in ["zero", "trailing", "price", "size"]:
    counts[f"{model}_within_1pip_pct"] = (
        100 * counts[f"{model}_hits"] / counts["rows"]
    )

counts["size_vs_zero_percentage_points"] = (
    100 * (counts["size_hits"] - counts["zero_hits"]) / counts["rows"]
)

order = {"ALL": 0, "normal": 1, "thin": 2, "stressed": 3, "unclassified": 4}
counts = counts.sort_values(
    "regime", key=lambda values: values.map(order)
).reset_index(drop=True)

overall_rows = int(counts.loc[counts["regime"].eq("ALL"), "rows"].iloc[0])
if int(counts.loc[counts["regime"].ne("ALL"), "rows"].sum()) != overall_rows:
    raise RuntimeError("Regime counts do not reconcile with the overall population.")

output = root / "reports/tables" / RUN / "within_one_pip_30s.csv"
output.parent.mkdir(parents=True, exist_ok=True)
counts.to_csv(output, index=False)

print("WITHIN_ONE_PIP_30S — final test")
print(f"Matched observations: {overall_rows:,}")
with pd.option_context("display.float_format", "{:,.4f}".format):
    display(counts[[
        "regime",
        "rows",
        "zero_within_1pip_pct",
        "price_within_1pip_pct",
        "size_within_1pip_pct",
        "trailing_within_1pip_pct",
        "size_vs_zero_percentage_points",
    ]])

print("PASS: forecast integrity, matched population, dates, and regime totals.")
print("Positive percentage-point difference means size beat zero.")
print("Saved:", output.relative_to(root))

WITHIN_ONE_PIP_30S — final test
Matched observations: 20,221,145


,regime,rows,zero_within_1pip_pct,price_within_1pip_pct,size_within_1pip_pct,trailing_within_1pip_pct,size_vs_zero_percentage_points
0,ALL,20221145,86.4345,85.6049,85.6025,75.1390,-0.8320
1,normal,16082483,85.1657,84.2368,84.2341,72.3470,-0.9316
2,thin,2695052,96.4451,96.0232,96.0210,92.8772,-0.4241
3,stressed,860973,71.8240,71.2258,71.2296,60.5789,-0.5943
4,unclassified,582637,96.7410,96.4249,96.4225,91.6715,-0.3186


PASS: forecast integrity, matched population, dates, and regime totals.
Positive percentage-point difference means size beat zero.
Saved: reports\tables\66d2e5a9ff903fe48113c6ea\within_one_pip_30s.csv


In [19]:
from pathlib import Path
import hashlib
import json

import duckdb
import pandas as pd
from IPython.display import display

RUN = "66d2e5a9ff903fe48113c6ea"
HORIZON = 60
TOLERANCE_PIPS = 1.0

root = next(
    (
        p.resolve()
        for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / f"artifacts/manifests/completion/{RUN}/final-test/evaluation.json"
        ).is_file()
    ),
    None,
)
if root is None:
    root = Path(
        input("Full repository folder path: ").strip().strip('"')
    ).expanduser().resolve()

run_directory = root / "artifacts/manifests/completion" / RUN
evaluation = json.loads(
    (run_directory / "final-test/evaluation.json").read_text(encoding="utf-8")
)
if evaluation["status"] != "complete":
    raise RuntimeError("Final-test evaluation is incomplete.")

expected_rows = next(
    row["matched_rows"]
    for row in evaluation["coverage"]
    if row["horizon_seconds"] == HORIZON
)

# Verify and select committed forecast files.
files = []
checkpoints = sorted(
    (run_directory / "final-test/evaluation_parts").glob("*.json")
)
if not checkpoints:
    raise RuntimeError("No committed forecast blocks found.")

for checkpoint in checkpoints:
    saved = json.loads(checkpoint.read_text(encoding="utf-8"))
    if saved["run_id"] != RUN or saved["fold"] != evaluation["fold"]:
        raise RuntimeError("Forecast checkpoint belongs to another evaluation.")

    record = saved["forecast"]
    path = (root / record["path"]).resolve()
    if not path.is_relative_to(root):
        raise RuntimeError("Forecast path escapes the repository.")

    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)

    if (
        path.stat().st_size != record["bytes"]
        or digest.hexdigest() != record["sha256"]
    ):
        raise RuntimeError(f"Forecast integrity failed: {path.name}")
    files.append(str(path))

models = ["zero", "trailing", "price", "size"]
source_columns = [
    f"zero_{HORIZON}s",
    f"trailing_{HORIZON}s",
    f"ridge_price_{HORIZON}s",
    f"ridge_price_size_{HORIZON}s",
]

connection = duckdb.connect(config={"memory_limit": "256MB", "threads": 2})
try:
    connection.read_parquet(
        files, hive_partitioning=False
    ).create_view("saved_forecasts")

    aliases = ", ".join(
        f"{column} AS {model}_forecast"
        for model, column in zip(models, source_columns)
    )
    connection.execute(f"""
        CREATE TEMP VIEW comparison AS
        SELECT
            feature_timestamp_utc,
            liquidity_regime,
            actual_{HORIZON}s AS actual,
            {aliases}
        FROM saved_forecasts
        WHERE eligible_{HORIZON}s
    """)

    invalid_conditions = " OR ".join(
        f"isfinite({name}) IS NOT TRUE"
        for name in ["actual", *[f"{model}_forecast" for model in models]]
    )
    checks = connection.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(*) FILTER (
                WHERE {invalid_conditions}
                   OR liquidity_regime IS NULL
                   OR liquidity_regime NOT IN
                      ('normal', 'thin', 'stressed', 'unclassified')
            ) AS invalid_rows,
            MAX(ABS(zero_forecast)) AS maximum_zero_forecast,
            MIN(feature_timestamp_utc) AS first_timestamp,
            MAX(feature_timestamp_utc) AS last_timestamp
        FROM comparison
    """).fetchdf().iloc[0]

    if int(checks["rows"]) != expected_rows or int(checks["invalid_rows"]) != 0:
        raise RuntimeError("Population reconciliation or forecast validity failed.")
    if checks["maximum_zero_forecast"] != 0:
        raise RuntimeError("Zero-change forecasts are unexpectedly nonzero.")

    start = pd.Timestamp(evaluation["fold"]["evaluation_start"])
    end = pd.Timestamp(evaluation["fold"]["evaluation_end"])
    if not (
        start <= pd.Timestamp(checks["first_timestamp"])
        <= pd.Timestamp(checks["last_timestamp"]) < end
    ):
        raise RuntimeError("Forecast dates fall outside the final test.")

    # Include exactly one pip; epsilon handles floating-point rounding only.
    limit = TOLERANCE_PIPS + 1e-9
    hit_expressions = ", ".join(
        f"ABS(actual - {model}_forecast) <= {limit} AS {model}_hit"
        for model in models
    )
    hit_counts = ", ".join(
        f"COUNT(*) FILTER (WHERE {model}_hit) AS {model}_hits"
        for model in models
    )

    counts = connection.execute(f"""
        WITH hits AS (
            SELECT liquidity_regime, {hit_expressions}
            FROM comparison
        )
        SELECT
            CASE WHEN GROUPING(liquidity_regime) = 1
                 THEN 'ALL' ELSE liquidity_regime END AS regime,
            COUNT(*) AS rows,
            {hit_counts},
            COUNT(*) FILTER (
                WHERE size_hit AND NOT zero_hit
            ) AS size_only_hits,
            COUNT(*) FILTER (
                WHERE zero_hit AND NOT size_hit
            ) AS zero_only_hits
        FROM hits
        GROUP BY GROUPING SETS ((), (liquidity_regime))
    """).fetchdf()
finally:
    connection.close()

for model in models:
    counts[f"{model}_within_1pip_pct"] = (
        100 * counts[f"{model}_hits"] / counts["rows"]
    )

counts["size_vs_zero_percentage_points"] = (
    100 * (counts["size_hits"] - counts["zero_hits"]) / counts["rows"]
)

order = {"ALL": 0, "normal": 1, "thin": 2, "stressed": 3, "unclassified": 4}
counts = counts.sort_values(
    "regime", key=lambda values: values.map(order)
).reset_index(drop=True)

overall_rows = int(counts.loc[counts["regime"].eq("ALL"), "rows"].iloc[0])
if int(counts.loc[counts["regime"].ne("ALL"), "rows"].sum()) != overall_rows:
    raise RuntimeError("Regime totals do not reconcile.")

output = root / "reports/tables" / RUN / "within_one_pip_60s.csv"
output.parent.mkdir(parents=True, exist_ok=True)
counts.to_csv(output, index=False)

print("WITHIN_ONE_PIP_60S — final test")
print(f"Matched observations: {overall_rows:,}")

with pd.option_context("display.float_format", "{:,.4f}".format):
    display(counts[[
        "regime",
        "rows",
        "zero_within_1pip_pct",
        "price_within_1pip_pct",
        "size_within_1pip_pct",
        "trailing_within_1pip_pct",
        "size_vs_zero_percentage_points",
    ]])

print("PASS: forecast integrity, matched population, dates, and regime totals.")
print("Positive percentage-point difference means size beat zero.")
print("Saved:", output.relative_to(root))

WITHIN_ONE_PIP_60S — final test
Matched observations: 19,152,419


,regime,rows,zero_within_1pip_pct,price_within_1pip_pct,size_within_1pip_pct,trailing_within_1pip_pct,size_vs_zero_percentage_points
0,ALL,19152419,74.7942,73.6586,73.6513,60.2835,-1.1429
1,normal,15914350,73.0698,71.8705,71.8617,57.3807,-1.2081
2,thin,2448234,90.2211,89.3535,89.3500,82.7804,-0.8712
3,stressed,789832,61.7203,61.0382,61.0494,49.0387,-0.6709
4,unclassified,3,100.0000,100.0000,100.0000,100.0000,0.0000


PASS: forecast integrity, matched population, dates, and regime totals.
Positive percentage-point difference means size beat zero.
Saved: reports\tables\66d2e5a9ff903fe48113c6ea\within_one_pip_60s.csv
